<a href="https://colab.research.google.com/github/Viren1212/Makhana-Work/blob/main/colab_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [169]:
# Import essential libraries grouped by functionality for clarity
# Image Processing and Computer Vision
import cv2                  # OpenCV for core image processing and computer vision operations
import numpy as np          # NumPy for efficient numerical computations and array manipulations
from sklearn.cluster import KMeans  # For ordering image using Kmeans



# Data Analysis and Visualization
import matplotlib.pyplot as plt  # Matplotlib for creating plots and visualizations
import pandas as pd         # Pandas for structured data handling and statistical analysis
import seaborn as sns       # Seaborn for advanced statistical visualizations (e.g., heatmaps)
from matplotlib.backends.backend_pdf import PdfPages  # Enables PDF report generation
from matplotlib.gridspec import GridSpec  # Advanced layout control for Matplotlib figures
import numpy as np

# Utilities and File Management
import math                 # Mathematical functions (e.g., sqrt, pi) for metric calculations
import os                   # Operating system utilities for file and directory operations
import json                 # JSON handling for calibration and metrics caching
import logging              # Logging for tracking events, errors, and process flow
from google.colab import files  # Google Colab-specific utility for file uploads/downloads
from collections import Counter  # Efficient counting of grade occurrences
from datetime import datetime  # Timestamp generation for unique identifiers and metadata
import pytz
IST = pytz.timezone('Asia/Kolkata')
def get_ist_now():
    return datetime.now(pytz.utc).astimezone(IST)


# Parallel Processing
import concurrent.futures   # Framework for concurrent task execution
from concurrent.futures import ProcessPoolExecutor, as_completed  # Tools for process-based parallelism


# Matplotlib Compatibility Handling
import matplotlib           # Base Matplotlib module for version checks
# Ensure colormap compatibility across Matplotlib versions
if hasattr(matplotlib, 'colormaps'):
    get_cmap = lambda name: matplotlib.colormaps[name]
else:
    from matplotlib.cm import get_cmap



In [170]:
# ===========================================================================
# Enhanced Fixed Setup Calibration System
# Purpose: Convert pixel measurements to real-world units (mm) for accurate grading
# ===========================================================================


# File path for storing calibration data
CALIBRATION_FILE = "enhanced_calibration.json"
SOFTWARE_VERSION = "v1.0"


# Schema to validate calibration data structure
CALIBRATION_SCHEMA = {
    "required": ["mm_per_pixel", "calibration_date"],  # Essential fields for all calibration methods
    "optional": ["camera_height_mm", "reference_diameter_mm", "method", "sensor_width", "focal_length"]  # Optional fields based on method
}


def validate_calibration_data(data):
    """
    Validates calibration data against the schema to ensure integrity.


    Args:
        data (dict): Calibration data dictionary to check.


    Returns:
        bool: True if data is valid.


    Raises:
        ValueError: If required fields are missing or geometric method lacks camera height.
    """
    for field in CALIBRATION_SCHEMA["required"]:
        if field not in data:
            raise ValueError(f"Missing required field: {field}")
    if data.get("method", "reference") == "geometric" and "camera_height_mm" not in data:
        raise ValueError("Missing required field 'camera_height_mm' for geometric calibration")
    return True


def calibrate_with_reference():
    """
    Calibrates pixel-to-mm conversion using a reference object (e.g., coin) in an image.


    Returns:
        float: Calibration factor (mm_per_pixel).


    Raises:
        FileNotFoundError: If reference image fails to load.
        ValueError: If no circles are detected.
    """
    logging.info("\n=== Starting Reference Object Calibration ===")
    uploaded = files.upload()  # Prompt user to upload an image with a reference object
    ref_path = list(uploaded.keys())[0]  # Extract the uploaded file path
    ref_image = cv2.imread(ref_path)  # Load the image
    if ref_image is None:
        logging.error("Failed to load reference image")
        raise FileNotFoundError("Failed to load reference image")
    gray_ref = cv2.cvtColor(ref_image, cv2.COLOR_BGR2GRAY)  # Convert to grayscale for processing
    # Detect circles using Hough Transform with tuned parameters
    circles = cv2.HoughCircles(
        gray_ref, cv2.HOUGH_GRADIENT, dp=1, minDist=100,
        param1=50, param2=30, minRadius=10, maxRadius=100
    )
    if circles is None or len(circles) == 0:
        logging.error("No circles detected in reference image")
        raise ValueError("No circles detected in reference image")
    circles = circles[0, :]  # Flatten circle data
    ref_circle = max(circles, key=lambda x: x[2])
    px_diameter = 2 * ref_circle[2]
    ref_object_name = input("Enter reference object name (e.g., Indian ₹10 Coin): ")
    actual_mm = float(input("Enter reference object diameter (mm): "))
    mm_per_pixel = actual_mm / px_diameter
    image_height, image_width = ref_image.shape[:2]
    logging.info(f"Calculated mm_per_pixel: {mm_per_pixel:.4f}")

    ref_image_path = "calibration_reference_image.png"
    cv2.imwrite(ref_image_path, ref_image)

    return {
        'mm_per_pixel': mm_per_pixel,
        'reference_object': ref_object_name,
        'actual_diameter_mm': actual_mm,
        'measured_diameter_px': round(px_diameter, 2),
        'image_resolution': f"{image_width} x {image_height}",
        'reference_image_path': ref_image_path
    }


def geometric_calibration():
    """
    Calibrates using camera geometry specifications provided by the user.


    Returns:
        tuple: (mm_per_pixel, camera_height_mm) calibration factor and height.


    Notes:
        Requires precise user inputs for accurate results.
    """
    logging.info("\n=== Starting Geometric Calibration ===")
    specs = {}  # Dictionary to store camera specifications
    # Prompt user for required camera parameters
    for param in [('sensor_width_mm', "sensor width (mm)"), ('focal_length_mm', "focal length (mm)"), ('image_width_px', "image width (pixels)")]:
        specs[param[0]] = float(input(f"Enter {param[1]}: "))
    camera_height = float(input("Enter camera height from base (mm): "))
    # Calculate mm_per_pixel using geometric formula
    mm_per_pixel = (specs['sensor_width_mm'] * camera_height) / (specs['focal_length_mm'] * specs['image_width_px'])
    logging.info(f"Calculated mm_per_pixel: {mm_per_pixel:.4f} with camera height: {camera_height} mm")
    return mm_per_pixel, camera_height


def save_calibration(data):
    """
    Saves calibration data to a JSON file for persistence.


    Args:
        data (dict): Calibration data to save.
    """
    # Convert NumPy types to native Python types for JSON compatibility
    for key, value in data.items():
        if isinstance(value, (np.generic,)):
            data[key] = value.item()
    # Add metadata for tracking
    data.update({
        "version": "1.1",
        "calibration_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })
    logging.info(f"Saving calibration: method={data['method']}, mm_per_pixel={data['mm_per_pixel']:.4f}")
    with open(CALIBRATION_FILE, 'w') as f:
        json.dump(data, f, indent=2)  # Write with indentation for readability
    logging.info(f"Calibration data saved to {CALIBRATION_FILE}")


def load_calibration():
    """
    Loads existing calibration data from file if it exists and is valid.


    Returns:
        dict or None: Calibration data if loaded successfully, None otherwise.
    """
    try:
        with open(CALIBRATION_FILE, 'r') as f:
            data = json.load(f)
        validate_calibration_data(data)  # Ensure data meets schema requirements
        logging.info(f"Loaded calibration: method={data['method']}, mm_per_pixel={data['mm_per_pixel']:.4f}, date={data['calibration_date']}")
        return data
    except (IOError, ValueError, KeyError) as e:
        logging.warning(f"Calibration loading failed: {str(e)}")
        return None


def calibration_menu():
    """
    Interactive menu to select and execute a calibration method.


    Returns:
        dict: Calibration data including method and mm_per_pixel.


    Raises:
        ValueError: If an invalid method is selected.
    """
    logging.info("\n=== Calibration Method Selection ===")
    print("1. Reference Object (Recommended)\n2. Geometric (Camera Specs)\n3. Manual Entry")
    choice = input("Select method (1-3): ")
    camera_model = input("Enter camera model (e.g., JAI AD-130GE): ")
    if choice == '1':
        ref_data = calibrate_with_reference()
        return {'method': 'reference', 'camera_model': camera_model, **ref_data}
    elif choice == '2':
        mm_px, cam_height = geometric_calibration()
        return {'method': 'geometric', 'camera_model': camera_model, 'mm_per_pixel': mm_px, 'camera_height_mm': cam_height}
    elif choice == '3':
        mm_per_pixel = float(input("Enter known mm/pixel value: "))
        return {'method': 'manual', 'camera_model': camera_model, 'mm_per_pixel': mm_per_pixel}
    logging.error("Invalid calibration method selected")
    raise ValueError("Invalid selection")

def get_calibration_factor(recalibrate=True):
    """
    Retrieves or computes the calibration factor based on user preference.


    Args:
        recalibrate (bool): If True, forces new calibration; if False, tries to load existing data.


    Returns:
        float: Calibration factor (mm_per_pixel).
    """
def get_calibration_factor(recalibrate=True):
    if not recalibrate:
        data = load_calibration()
        if data:
            logging.info(f"Using loaded {data['method']} calibration from {data['calibration_date']}")
            return data['mm_per_pixel'], data
    logging.info("No valid calibration found or recalibration requested. Starting new calibration...")
    calib_data = calibration_menu()
    calib_data['calibration_date'] = get_ist_now().strftime("%Y-%m-%d")
    calib_data['calibration_status'] = 'Successful'
    save_calibration(calib_data)
    return calib_data['mm_per_pixel'], calib_data

In [171]:
# ===========================================================================
# Configurable Thresholds
# Purpose: Define criteria for filtering and grading makhanas
# ===========================================================================


# Thresholds for processing and grading makhanas
THRESHOLDS = {
    'diameter_mm': 40.0,          # Maximum allowed diameter in mm for grading
    'circularity': 0.3,           # Minimum circularity score to filter objects
    'aspect_ratio': 0.3,          # Minimum aspect ratio to consider valid objects
    'min_contour_area': 1000,     # Minimum contour area (pixels) to exclude noise
    'histogram_bins': 20,         # Number of bins for histogram-based color analysis
    'min_white_percentage': 50.0, # Minimum white percentage for quality grading
    'min_whiteness': 25.0         # Minimum whiteness (L* value) for grading
}


In [172]:
# ===========================================================================
# Batch Grading System with Weighted Scoring
# Purpose: Assign grades based on size, shape, and color metrics
# ===========================================================================


# Grade Driteria for makhanas based on multiple metrics
BATCH_GRADES = {
    'Grade A': {
        'min_diameter': 25, 'max_diameter': 60,
        'min_circularity': 0.9, 'max_circularity': 1.0,
        'min_aspect_ratio': 0.9, 'max_aspect_ratio': 1.0,
        'min_white_percentage': 90.0, 'max_white_percentage': 100.0,
        'min_whiteness': 50.0, 'max_whiteness': 100.0
    },
    'Grade B': {
        'min_diameter': 20, 'max_diameter': 25,
        'min_circularity': 0.8, 'max_circularity': 0.9,
        'min_aspect_ratio': 0.7, 'max_aspect_ratio': 0.9,
        'min_white_percentage': 80.0, 'max_white_percentage': 90.0,
        'min_whiteness': 40.0, 'max_whiteness': 50.0
    },
    'Grade C': {
        'min_diameter': 15, 'max_diameter': 20,
        'min_circularity': 0.6, 'max_circularity': 0.8,
        'min_aspect_ratio': 0.5, 'max_aspect_ratio': 0.7,
        'min_white_percentage': 70.0, 'max_white_percentage': 80.0,
        'min_whiteness': 35.0, 'max_whiteness': 40.0
    },
    'Grade D': {
        'min_diameter': 10, 'max_diameter': 15,
        'min_circularity': 0.4, 'max_circularity': 0.6,
        'min_aspect_ratio': 0.3, 'max_aspect_ratio': 0.5,
        'min_white_percentage': 60, 'max_white_percentage': 70.0,
        'min_whiteness': 20.0, 'max_whiteness': 35.0
    }
}


# Weights for metrics in the grading system (sum to 1.0 for balanced scoring)
METRIC_WEIGHTS = {
    'Effective Diameter (mm)': 0.25,
    'Circularity': 0.25,
    'Aspect Ratio': 0.1,
    'White Percentage': 0.2,
    'Whiteness': 0.2
}


# Mapping to compare each Grade Bgainst the next higher grade
COMPARISON_GRADE = {
    'Grade B': 'Grade A',
    'Grade C': 'Grade B',
    'Grade D': 'Grade C'
}


def normalize_metric(value, min_val, max_val):
    """
    Normalizes a metric value to a 0-1 range for weighted scoring.


    Args:
        value (float): Metric value to normalize.
        min_val (float): Minimum acceptable value.
        max_val (float): Maximum acceptable value.


    Returns:
        float: Normalized score between 0 and 1.
    """
    if max_val == min_val:  # Avoid division by zero
        return 1.0 if value >= min_val else 0.0
    return max(0.0, min(1.0, (value - min_val) / (max_val - min_val)))


def determine_grade(item_metrics):
    """
    Assigns a Grade Cased on weighted scoring of item metrics.


    Args:
        item_metrics (dict): Metrics for a single makhana (diameter, circularity, etc.).


    Returns:
        str: Assigned grade ('Grade A', 'Grade B', 'Grade C', 'Grade D').
    """
    for grade in ['Grade A', 'Grade B', 'Grade C']:
        specs = BATCH_GRADES[grade]
        total_score = 0.0
        for metric, weight in METRIC_WEIGHTS.items():
            base_key = 'diameter' if metric == 'Effective Diameter (mm)' else metric.lower().replace(' ', '_')
            min_key = f'min_{base_key}'
            max_key = f'max_{base_key}'
            min_val = specs[min_key]
            max_val = specs[max_key]
            value = item_metrics[metric]
            normalized = normalize_metric(value, min_val, max_val)
            total_score += weight * normalized
        if total_score >= 0.8:  # Threshold for Grade Bssignment
            logging.debug(f"Assigned grade {grade} with score {total_score:.2f}")
            return grade
    logging.debug("Assigned grade 'Grade D' (score below threshold)")
    return "Grade D"


def calculate_grade_distribution(metrics):
    """
    Computes the distribution of grades across a batch.


    Args:
        metrics (list): List of metric dictionaries for all makhanas.


    Returns:
        dict: Grade distribution with counts and percentages.
    """
    grade_counts = {grade: 0 for grade in BATCH_GRADES}
    for m in metrics:
        grade = m['Grade']
        grade_counts[grade] += 1
    total = len(metrics)
    dist = {grade: {'count': count, 'percentage': (count / total * 100) if total > 0 else 0}
            for grade, count in grade_counts.items()}
    logging.info(f"Grade distribution: {dist}")
    return dist

In [173]:
# ===========================================================================
# Color Analysis Functions
# Purpose: Evaluate color properties (whiteness) of makhanas
# ===========================================================================


def analyze_color(image, mask):
    """
    Analyzes color properties of a segmented makhana to assess whiteness.


    Args:
        image (numpy.ndarray): Input image in BGR format.
        mask (numpy.ndarray): Binary mask of the makhana.


    Returns:
        tuple: (white_percentage, whiteness) - adjusted white area percentage and average lightness.
    """
    try:
        # Convert to Lab color space for accurate lightness (L) measurement
        lab_image = cv2.cvtColor(image, cv2.COLOR_BGR2Lab)
        l_channel = lab_image[:, :, 0]  # Extract L channel (lightness)
        # Erode mask to exclude edge artifacts
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        core_mask = cv2.erode(mask, kernel, iterations=1)
        # Apply mask to L channel
        masked_l = cv2.bitwise_and(l_channel, l_channel, mask=core_mask)
        # Identify dark spots (L < 70)
        dark_spot_mask = ((masked_l < 70) & (core_mask > 0)).astype(np.uint8) * 255
        # Clean noise in dark spot detection
        dark_clean = cv2.morphologyEx(dark_spot_mask, cv2.MORPH_OPEN, kernel)
        dark_clean = cv2.morphologyEx(dark_clean, cv2.MORPH_CLOSE, kernel)
        # Ignore tiny dark areas (< 0.2% of total area)
        dark_ratio = cv2.countNonZero(dark_clean) / (cv2.countNonZero(core_mask) + 1e-5)
        if dark_ratio < 0.002:
            dark_clean[:] = 0
        # Define white region by excluding dark spots
        white_region_mask = cv2.bitwise_and(core_mask, cv2.bitwise_not(dark_clean))
        # Calculate whiteness as mean L value of white region
        white_region_pixels = masked_l[white_region_mask > 0]
        whiteness = np.mean(white_region_pixels) if white_region_pixels.size > 0 else 0
        # Calculate white percentage with adjustment
        total_area = cv2.countNonZero(core_mask)
        white_area = cv2.countNonZero(white_region_mask)
        white_percentage = ((white_area / total_area) * 100) if total_area > 0 else 0
        # Adjust and scale outputs
        # white_percentage += 10  # Adjustment factor
        whiteness = (whiteness / 255) * 100  # Scale to 0-100
        return white_percentage, whiteness
    except Exception as e:
        logging.error(f"Error in color analysis: {str(e)}")
        return 0, 0


In [174]:
import numpy as np
# ===========================================================================
# Process a Single Image with Error Handling
# Purpose: Extract metrics and visualizations from one image
# ===========================================================================


def process_single_image(image, mm_per_pixel, thresholds={"min_contour_area": 100}):
    """
    Processes a single image to detect makhanas and compute their metrics.


    Args:
        image (numpy.ndarray): Input image in BGR format.
        mm_per_pixel (float): Calibration factor for pixel-to-mm conversion.
        thresholds (dict): Filtering thresholds (e.g., min_contour_area).


    Returns:
        tuple: (makhana_metrics, image_with_contours, segmentation_vis, individual_masks).
    """
    try:
        if image is None or image.size == 0:
            logging.error("Input image is None or empty")
            raise ValueError("Input image is None or empty")
        # --- Preprocessing ---
        image_with_contours = image.copy()  # Copy for contour overlay
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # Grayscale conversion
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)  # Reduce noise
        # Adaptive thresholding with Otsu, adjusted for faint edges
        otsu_val, _ = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        adjusted_thresh = max(0, int(otsu_val - 15))  # Lower threshold for better detection
        _, binary = cv2.threshold(blurred, adjusted_thresh, 255, cv2.THRESH_BINARY)
        # Dilate to connect object parts
        kernel = np.ones((5, 5), np.uint8)
        dilated = cv2.dilate(binary, kernel, iterations=2)
        # Refine with distance transform
        dist = cv2.distanceTransform(dilated, cv2.DIST_L2, 5)
        _, refined = cv2.threshold(dist, 1.5, 255, 0)
        refined = refined.astype(np.uint8)
        # Segment objects via connected components
        num_labels, markers = cv2.connectedComponents(refined)
        unique_labels = list(range(1, num_labels))  # Exclude background
        # --- Metric Extraction ---
        makhana_metrics = []
        individual_masks = []
        detected_objects = []      # Step 1 Change
        for label in unique_labels:
            mask = (markers == label).astype(np.uint8) * 255
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if not contours:
                continue
            cnt = max(contours, key=cv2.contourArea)

            points = cnt[:, 0, :]      # Convert contour to Nx2 array

            max_dist = 0

            feret_pt1 = None
            feret_pt2 = None

            for i in range(len(points)):
                for j in range(i + 1, len(points)):
                    d = np.linalg.norm(points[i] - points[j])

                    if d > max_dist:
                        max_dist = d
                        feret_pt1 = tuple(points[i])
                        feret_pt2 = tuple(points[j])

            max_feret_mm = max_dist * mm_per_pixel

            M = cv2.moments(cnt)

            if M["m00"] == 0:
                continue

            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            area_pixels = cv2.contourArea(cnt)
            if area_pixels < thresholds['min_contour_area']:
                continue
            # Smooth contour for better perimeter measurement
            epsilon = 0.005 * cv2.arcLength(cnt, True)
            cnt_smooth = cv2.approxPolyDP(cnt, epsilon, True)
            perimeter_pixels = cv2.arcLength(cnt_smooth, True)
            # Calculate circularity with enhancement
            circularity = min(1.0, (4 * math.pi * area_pixels) / (perimeter_pixels ** 2)) if perimeter_pixels > 0 else 0
            #circularity = circularity ** 3  # Amplify differences
            # Compute aspect ratio using fitted ellipse
            if len(cnt) >= 5:
                ellipse = cv2.fitEllipse(cnt)
                major_axis, minor_axis = max(ellipse[1]), min(ellipse[1])
                aspect_ratio = minor_axis / major_axis if major_axis > 0 else 0
            else:
                aspect_ratio = 0

            # Perform color analysis
            full_mask = np.zeros_like(gray)
            cv2.drawContours(full_mask, [cnt], -1, 255, -1)
            white_percentage, whiteness = analyze_color(image, full_mask)
            # Compile metrics
            metrics = {
                "Original Label": label, # OPEN CV LABEL [Step 6 Change]
                "Label": label, #Will later become row wise label
                "Effective Diameter (mm)": math.sqrt((4 * area_pixels) / math.pi) * mm_per_pixel,
                "Circularity": circularity,
                "Aspect Ratio": aspect_ratio,
                "Max Feret Diameter (mm)": max_feret_mm,
                "White Percentage": white_percentage,
                "Whiteness": whiteness
            }
            metrics["Grade"] = determine_grade(metrics)
            # Extract individual makhana image and mask
            x, y, w, h = cv2.boundingRect(cnt)
            padding = 10
            x_start = max(0, x - padding)
            y_start = max(0, y - padding)
            x_end = min(image.shape[1], x + w + padding)
            y_end = min(image.shape[0], y + h + padding)
            individual_image = image[y_start:y_end, x_start:x_end].copy()
            individual_mask = np.zeros((y_end - y_start, x_end - x_start), dtype=np.uint8)
            shifted_contour = cnt.copy()
            shifted_contour[:, :, 0] -= x_start
            shifted_contour[:, :, 1] -= y_start
            cv2.drawContours(individual_mask, [shifted_contour], -1, 255, -1)

            #STEP 2 Chage
            detected_objects.append({
              "cx": cx,
              "cy": cy,
              "image": individual_image,
              "mask": individual_mask,
              "metrics": metrics
          })


       # STEP 3: Ask for number of rows
        num_rows = int(input("Enter number of rows: "))

        # STEP 4: Cluster objects into rows using KMeans on cy
        y_values = np.array([[obj["cy"]] for obj in detected_objects])

        kmeans = KMeans(n_clusters=num_rows, random_state=42, n_init=10)
        kmeans.fit(y_values)

        for obj, row_id in zip(detected_objects, kmeans.labels_):
            obj["row"] = row_id

        # Re-rank cluster IDs so row 0 = topmost row
        cluster_centers = kmeans.cluster_centers_.flatten()
        sorted_cluster_order = np.argsort(cluster_centers)
        cluster_id_to_row_number = {
            old_id: new_row for new_row, old_id in enumerate(sorted_cluster_order)
        }
        for obj in detected_objects:
            obj["row"] = cluster_id_to_row_number[obj["row"]]

        # STEP 5: Create rows explicitly
        rows = [[] for _ in range(num_rows)]

        for obj in detected_objects:
            rows[obj["row"]].append(obj)

        # Sort each row from left to right
        for row in rows:
            row.sort(key=lambda obj: obj["cx"])
        print("Rows found:", len(rows))
        for r in rows:
            print([obj["cy"] for obj in r])
        # STEP 5 Change
        label = 1

        for row_idx, row in enumerate(rows):
            for obj in row:
                obj["metrics"]["Label"] = label
                obj["metrics"]["Row"] = row_idx

                makhana_metrics.append(obj["metrics"])

                individual_masks.append({
                    "image": obj["image"],
                    "mask": obj["mask"],
                    "metrics": obj["metrics"]
                })

                label += 1
        # print("Final Labels:")

        # for row in rows:
        #     print([obj["metrics"]["Label"] for obj in row])
        # print("\nRows:")
        # for row in rows:
        #     print([(obj["metrics"]["Label"], obj["cx"], obj["cy"]) for obj in row])

        # --- Visualization ---
        grade_colors = {
            "Grade A": (0, 255, 0),   # Green
            "Grade B": (0, 255, 255), # Cyan
            "Grade C": (0, 165, 255), # Orange-ish
            "Grade D": (0, 0, 255)    # Red
        }
        # STEP 7 Change
        label_to_grade = {m["Original Label"]: m["Grade"] for m in makhana_metrics}
        segmentation_vis = np.zeros_like(image)
        for label in unique_labels:
            grade = label_to_grade.get(label, "Grade D")
            color = grade_colors.get(grade, (255, 255, 255))  # Default white
            segmentation_vis[markers == label] = color
        logging.info(f"Processed image: detected {len(makhana_metrics)} makhanas")
        return makhana_metrics, image_with_contours, segmentation_vis, individual_masks
    except Exception as e:
        logging.error(f"Error processing image: {str(e)}")
        blank_image = np.zeros_like(image) if image is not None else np.zeros((100, 100, 3), dtype=np.uint8)
        return [], blank_image, blank_image, []

In [175]:
# ===========================================================================
# Batch Processing with Memory Management
# Purpose: Efficiently process multiple images in parallel
# ===========================================================================


def process_batch_images(image_paths, mm_per_pixel, batch_size=5, thresholds=THRESHOLDS):
    """
    Processes a batch of images in parallel with memory-efficient batching.


    Args:
        image_paths (list): List of image file paths.
        mm_per_pixel (float): Calibration factor.
        batch_size (int): Number of images per processing batch (default: 5).
        thresholds (dict): Processing thresholds.


    Returns:
        tuple: (all_metrics, all_image_with_contours, all_segmentation_vis, all_individual_masks).
    """
    all_metrics = []
    all_image_with_contours = []
    all_segmentation_vis = []
    all_individual_masks = []
    # Process images in batches to manage memory
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i + batch_size]
        images = [cv2.imread(path) for path in batch_paths]
        with ProcessPoolExecutor() as executor:
            futures = [executor.submit(process_single_image, img, mm_per_pixel, thresholds) for img in images]
            for future in as_completed(futures):
                try:
                    metrics, contours, seg_vis, masks = future.result()
                    all_metrics.append(metrics)
                    all_image_with_contours.append(contours)
                    all_segmentation_vis.append(seg_vis)
                    all_individual_masks.append(masks)
                except Exception as e:
                    logging.error(f"Error in batch processing: {str(e)}")
                    all_metrics.append([])
                    all_image_with_contours.append(np.zeros((100, 100, 3), dtype=np.uint8))
                    all_segmentation_vis.append(np.zeros((100, 100, 3), dtype=np.uint8))
                    all_individual_masks.append([])
    return all_metrics, all_image_with_contours, all_segmentation_vis, all_individual_masks



In [176]:
# ===========================================================================
# Caching Intermediate Results
# Purpose: Optimize performance by reusing computed metrics
# ===========================================================================


def convert_to_serializable(obj):
    """
    Converts objects to JSON-serializable formats.


    Args:
        obj: Object to convert (e.g., NumPy types, arrays, dicts).


    Returns:
        JSON-compatible version of the object.
    """
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    return obj


def save_metrics(cache_dir, sample_num, metrics):
    """
    Saves metrics to a JSON file in the cache directory.


    Args:
        cache_dir (str): Directory for storing cached files.
        sample_num (int): Sample identifier for file naming.
        metrics (list): Metrics to cache.
    """
    os.makedirs(cache_dir, exist_ok=True)
    cache_file = os.path.join(cache_dir, f"sample_{sample_num}.json")
    try:
        serializable_metrics = convert_to_serializable(metrics)
        with open(cache_file, 'w') as f:
            json.dump(serializable_metrics, f, indent=2)
        logging.info(f"Cached metrics for sample {sample_num} to {cache_file}")
    except Exception as e:
        logging.error(f"Failed to save metrics for sample {sample_num}: {str(e)}")


def load_metrics(cache_dir, sample_num):
    """
    Loads cached metrics from a file if available.


    Args:
        cache_dir (str): Directory containing cached files.
        sample_num (int): Sample identifier.


    Returns:
        list or None: Cached metrics or None if unavailable.
    """
    cache_file = os.path.join(cache_dir, f"sample_{sample_num}.json")
    if os.path.exists(cache_file):
        try:
            with open(cache_file, 'r') as f:
                metrics = json.load(f)
            logging.info(f"Loaded cached metrics for sample {sample_num} from {cache_file}")
            return metrics
        except Exception as e:
            logging.error(f"Failed to load cache for sample {sample_num}: {str(e)}")
            return None
    return None


In [177]:
def generate_batch_report(all_metrics, sample_images, segmentation_images, individual_masks_per_sample, mm_per_pixel, thresholds, num_samples, calib_data=None, report_metadata=None):
    """
    Generates a multi-page PDF report with analysis results and visualizations.


    Args:
        all_metrics (list): Metrics for all makhanas across samples.
        sample_images (list): Images with contours.
        segmentation_images (list): Segmentation visualizations.
        individual_masks_per_sample (list): Individual makhana data per sample.
        mm_per_pixel (float): Calibration factor.
        thresholds (dict): Processing thresholds.
        num_samples (int): Number of samples analyzed.
    """
    try:
        def add_watermark_and_footer(fig):
            """Adds a watermark and footer to each report page."""
            fig.text(0.5, 0.5, "CSIR-CSIO", fontsize=50, color='gray', ha='center', va='center', alpha=0.5, rotation=45)
            footer_line1 = "Report generated by Automated Makhana Grading System developed by CSIR-CSIO"
            if report_metadata:
                footer_line2 = f"v{report_metadata.get('Software Version', 'N/A')} | Operator: {report_metadata.get('Operator', 'N/A')} | {report_metadata.get('Date', '')} {report_metadata.get('Time', '')}"
                fig.text(0.5, 0.015, footer_line1, fontsize=8, ha='center', va='bottom')
                fig.text(0.5, 0.002, footer_line2, fontsize=6, ha='center', va='bottom', color='gray')
            else:
                fig.text(0.5, 0.01, footer_line1, fontsize=8, ha='center', va='bottom')


        # Formatting specifications for metrics display
        format_dict = {
            "Effective Diameter (mm)": ".2f",
            "Circularity": ".3f",
            "Aspect Ratio": ".3f",
            "Max Feret Diameter (mm)": ".2f",
            "White Percentage": ".1f",
            "Whiteness": ".1f"
        }

        # Create a PDF file
        pdf_filename = f"Makhana_Analysis_Report_{get_ist_now().strftime('%Y%m%d_%H%M%S')}.pdf"
        with PdfPages(pdf_filename) as pdf:

            # --- Page 1: Report Overview ---
            fig = plt.figure(figsize=(8.27, 11.69), dpi=100) # A4 size
            gs = GridSpec(6, 1, figure=fig, height_ratios=[1, 1, 1, 1, 1, 1])
            fig.suptitle('Makhana Quality Analysis Report', fontsize=16, fontweight='bold')

            # Overall Report Info
            ax0 = fig.add_subplot(gs[0, 0])
            ax0.axis('off')
            info_text = """\nReport Details:\n"""
            if report_metadata:
                for key, value in report_metadata.items():
                    info_text += f"{key}: {value}\n"
            info_text += f"Total Makhanas Analyzed: {len(all_metrics)}\n"
            info_text += f"Thresholds: {json.dumps(thresholds, indent=2)}\n" # Display thresholds
            ax0.text(0.05, 0.95, info_text, va='top', ha='left', fontsize=9, transform=ax0.transAxes)

            add_watermark_and_footer(fig)
            pdf.savefig(fig)
            plt.close(fig)

            # --- Page 2: Batch Summary Statistics ---
            fig = plt.figure(figsize=(8.27, 11.69), dpi=100)
            gs = GridSpec(3, 1, figure=fig, height_ratios=[0.5, 1, 1])
            fig.suptitle('Batch Summary Statistics', fontsize=16, fontweight='bold')

            # Convert metrics to DataFrame for easier statistics calculation
            df_metrics = pd.DataFrame(all_metrics)

            # Overall Batch Grade (using existing logic from Excel report)
            grade_points = {'Grade A': 100, 'Grade B': 66.666, 'Grade C': 33.333, 'Grade D': 0}
            total_points = sum(grade_points.get(m.get('Grade', 'Grade D'), 0) for m in all_metrics)
            average_score = total_points / len(all_metrics) if len(all_metrics) > 0 else 0

            if average_score >= 80:
                batch_grade = 'Grade A'
            elif average_score >= 60:
                batch_grade = 'Grade B'
            elif average_score >= 40:
                batch_grade = 'Grade C'
            else:
                batch_grade = 'Grade D'

            ax_batch_grade = fig.add_subplot(gs[0, 0])
            ax_batch_grade.axis('off')
            ax_batch_grade.text(0.5, 0.5, f"Overall Batch Grade: {batch_grade} (Average Score: {average_score:.2f})",
                                  fontsize=12, ha='center', va='center', fontweight='bold', transform=ax_batch_grade.transAxes)

            # Summary Statistics Table
            metrics_for_stats = [
                "Effective Diameter (mm)", "Circularity", "Aspect Ratio",
                "Max Feret Diameter (mm)", "White Percentage", "Whiteness"
            ]
            stats_data = []
            for metric in metrics_for_stats:
                if metric in df_metrics.columns:
                    values = df_metrics[metric].dropna()
                    if not values.empty:
                        stats_data.append({
                            'Metric': metric,
                            'Mean': f"{values.mean():{format_dict.get(metric, '.2f')}}",
                            'Std Dev': f"{values.std():{format_dict.get(metric, '.2f')}}",
                            'Min': f"{values.min():{format_dict.get(metric, '.2f')}}",
                            'Max': f"{values.max():{format_dict.get(metric, '.2f')}}"
                        })
            df_stats = pd.DataFrame(stats_data)

            ax1 = fig.add_subplot(gs[1, 0])
            ax1.axis('off')
            if not df_stats.empty:
                table = ax1.table(cellText=df_stats.values, colLabels=df_stats.columns, loc='center', cellLoc='center')
                table.auto_set_font_size(False)
                table.set_fontsize(8)
                table.scale(1.2, 1.2)
                ax1.set_title('Metric Summary Statistics', fontsize=12, pad=10)
            else:
                ax1.text(0.5, 0.5, 'No statistics available', va='center', ha='center', fontsize=10, transform=ax1.transAxes)

            # Grade Distribution Plot
            grade_dist = calculate_grade_distribution(all_metrics)
            grade_labels = list(grade_dist.keys())
            grade_percentages = [grade_dist[g]['percentage'] for g in grade_labels]

            ax2 = fig.add_subplot(gs[2, 0])
            if grade_percentages:
                ax2.bar(grade_labels, grade_percentages, color=['green', 'cyan', 'orange', 'red'])
                ax2.set_title('Grade Distribution', fontsize=12)
                ax2.set_ylabel('Percentage (%)')
                ax2.set_xlabel('Grade')
                for i_grade, v in enumerate(grade_percentages):
                    ax2.text(i_grade, v + 1, f"{v:.1f}%", ha='center', va='bottom', fontsize=8)
            else:
                ax2.text(0.5, 0.5, 'No grade distribution available', va='center', ha='center', fontsize=10, transform=ax2.transAxes)

            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            add_watermark_and_footer(fig)
            pdf.savefig(fig)
            plt.close(fig)

            # --- Pages 3+: Individual Sample Visualizations ---
            for i in range(num_samples):
                # Initialize fig and gs for the first page of individual makhanas for this sample
                fig = plt.figure(figsize=(8.27, 11.69), dpi=100)
                gs_sample_overview = GridSpec(4, 2, figure=fig, height_ratios=[1, 1, 1, 1]) # gs for sample overview + first set of makhanas
                fig.suptitle(f'Sample {i+1} Analysis', fontsize=16, fontweight='bold')

                # Original Image with Contours
                ax_img = fig.add_subplot(gs_sample_overview[0, :])
                if sample_images[i] is not None: # Ensure image data exists
                    ax_img.imshow(cv2.cvtColor(sample_images[i], cv2.COLOR_BGR2RGB))
                    ax_img.set_title(f'Sample {i+1} with Detected Makhanas', fontsize=12)
                else:
                    ax_img.text(0.5, 0.5, 'Image not available', va='center', ha='center', fontsize=10, transform=ax_img.transAxes)
                ax_img.axis('off')

                # Segmentation Visualization
                ax_seg = fig.add_subplot(gs_sample_overview[1, :])
                if segmentation_images[i] is not None: # Ensure segmentation data exists
                    ax_seg.imshow(cv2.cvtColor(segmentation_images[i], cv2.COLOR_BGR2RGB))
                    ax_seg.set_title(f'Sample {i+1} Segmentation (Color-coded Grades)', fontsize=12)
                else:
                    ax_seg.text(0.5, 0.5, 'Segmentation not available', va='center', ha='center', fontsize=10, transform=ax_seg.transAxes)
                ax_seg.axis('off')

                # Individual Makhana Visualizations and Metrics
                sample_individual_masks = individual_masks_per_sample[i]
                num_makhanas = len(sample_individual_masks)
                if num_makhanas > 0:
                    makhanas_per_page = 4
                    for j in range(0, num_makhanas, makhanas_per_page):
                        if j == 0: # First page of individual makhanas for this sample (after overview images)
                            current_gs = gs_sample_overview
                            row_offset = 2 # Start makhanas from the 3rd row (index 2)
                        else: # Subsequent pages for individual makhanas, fill the whole page
                            fig = plt.figure(figsize=(8.27, 11.69), dpi=100)
                            # Create a GridSpec specifically for makhanas (2x2 for 4 makhanas)
                            current_gs = GridSpec(makhanas_per_page // 2, 2, figure=fig) # This will be GridSpec(2,2)
                            fig.suptitle(f'Sample {i+1} Analysis (Continued)', fontsize=16, fontweight='bold')
                            row_offset = 0 # Makhanas start from row 0 on these new pages

                        for k in range(min(makhanas_per_page, num_makhanas - j)):
                            makhana_idx = j + k
                            makhana_data = sample_individual_masks[makhana_idx]
                            metrics = makhana_data['metrics']
                            makhana_img = makhana_data['image']

                            row_idx = (k // 2) + row_offset
                            col_idx = k % 2

                            ax = fig.add_subplot(current_gs[row_idx, col_idx])
                            if makhana_img is not None and makhana_img.size > 0:
                                ax.imshow(cv2.cvtColor(makhana_img, cv2.COLOR_BGR2RGB))
                            ax.axis('off')
                            title = f"ID: {metrics.get('Label', 'N/A')}"
                            title += f"\nGrade: {metrics.get('Grade', 'N/A')}"
                            title += f"\nDiameter: {metrics.get('Effective Diameter (mm)', 0.0):.2f}mm"
                            ax.set_title(title, fontsize=8)

                        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
                        add_watermark_and_footer(fig)
                        pdf.savefig(fig)
                        plt.close(fig)
                else:
                    # If there are no individual makhanas to display, ensure we still save the sample's overview page
                    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
                    add_watermark_and_footer(fig)
                    pdf.savefig(fig)
                    plt.close(fig)

            logging.info(f"PDF report generated successfully: {pdf_filename}")
            files.download(pdf_filename)

    except Exception as e:
        logging.error(f"Error generating PDF report: {str(e)}")

In [178]:
def generate_batch_excel(all_metrics, num_samples, mm_per_pixel, report_metadata=None):
    """
    Generates an Excel report with four sheets:
    1. 'Makhana Data'      - one row per makhana with all metrics
    2. 'Grade Summary'     - grade distribution counts and percentages
    3. 'Batch Summary'     - min/max/mean/median/std for each parameter
    4. 'Report Info'       - calibration, totals, and overall batch grade

    Args:
        all_metrics (list): Metrics for all makhanas across samples.
        num_samples (int): Number of samples analyzed.
        mm_per_pixel (float): Calibration factor used for this run.
    """
    try:
        excel_name = f"Makhana_Batch_Report_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

        # --- Sheet 1: Makhana Data ---
        columns = [
            "Sample", "Row", "Label", "Grade",
            "Effective Diameter (mm)", "Circularity", "Aspect Ratio", "Max Feret Diameter (mm)",
            "White Percentage", "Whiteness"
        ]
        rows_for_df = [{col: m.get(col, "") for col in columns} for m in all_metrics]
        df_data = pd.DataFrame(rows_for_df, columns=columns)
        df_data.sort_values(by=["Sample", "Label"], inplace=True)

        # --- Sheet 2: Grade Summary ---
        grade_dist = calculate_grade_distribution(all_metrics)
        summary_rows = [
            {"Grade": grade, "Count": d["count"], "Percentage": round(d["percentage"], 1)}
            for grade, d in grade_dist.items()
        ]
        df_summary = pd.DataFrame(summary_rows, columns=["Grade", "Count", "Percentage"])

        # --- Sheet 3: Batch Summary Stats ---
        params = ["Effective Diameter (mm)", "Circularity", "Aspect Ratio", "Max Feret Diameter (mm)", "White Percentage", "Whiteness"]
        stats_rows = []
        for param in params:
            values = [m[param] for m in all_metrics if param in m and not np.isnan(m[param])]
            if values:
                stats_rows.append({
                    "Parameter": param,
                    "Min": round(min(values), 3),
                    "Max": round(max(values), 3),
                    "Mean": round(np.mean(values), 3),
                    "Median": round(np.percentile(values, 50), 3),
                    "Std": round(np.std(values), 3)
                })
        df_stats = pd.DataFrame(stats_rows, columns=["Parameter", "Min", "Max", "Mean", "Median", "Std"])

        # --- Sheet 4: Report Info ---
        grade_points = {'Grade A': 100, 'Grade B': 66.666, 'Grade C': 33.333, 'Grade D': 0}
        total_points = sum(grade_points.get(m.get('Grade', 'Grade D'), 0) for m in all_metrics)
        average_score = total_points / len(all_metrics) if len(all_metrics) > 0 else 0
        if average_score >= 80:
            batch_grade = 'Grade A'
        elif average_score >= 60:
            batch_grade = 'Grade B'
        elif average_score >= 40:
            batch_grade = 'Grade C'
        else:
            batch_grade = 'Grade D'

        info_rows = [
            {"Field": "Report Generated", "Value": get_ist_now().strftime('%Y-%m-%d %H:%M')},
            {"Field": "Total Samples", "Value": num_samples},
            {"Field": "Total Makhanas Analyzed", "Value": len(all_metrics)},
            {"Field": "Calibration (mm/pixel)", "Value": round(mm_per_pixel, 4)},
            {"Field": "Overall Batch Grade", "Value": batch_grade},
            {"Field": "Average Score", "Value": round(average_score, 2)},
        ]
        if report_metadata:
            for k, v in report_metadata.items():
                info_rows.append({"Field": k, "Value": v})
        df_info = pd.DataFrame(info_rows, columns=["Field", "Value"])

        # --- Write all sheets to one Excel file ---
        with pd.ExcelWriter(excel_name, engine="openpyxl") as writer:
            df_data.to_excel(writer, sheet_name="Makhana Data", index=False)
            df_summary.to_excel(writer, sheet_name="Grade Summary", index=False)
            df_stats.to_excel(writer, sheet_name="Batch Summary", index=False)
            df_info.to_excel(writer, sheet_name="Report Info", index=False)

        logging.info(f"Excel report generated: {excel_name}")
        files.download(excel_name)
    except Exception as e:
        logging.error(f"Error generating Excel report: {str(e)}")

In [179]:

# ===========================================================================
# Batch Analysis Function
# Purpose: Orchestrate the entire analysis workflow
# ===========================================================================


def analyze_batch():
    """
    Manages the complete batch analysis process from image upload to report generation.


    Ensures all images and visualizations are included in the final report.
    """
    logging.info("Starting batch analysis")
    operator_name = input("Enter operator name: ")
    try:
        num_samples = int(input("Enter the number of samples to analyze: "))
        if num_samples <= 0:
            raise ValueError("Number of samples must be positive")
    except ValueError as e:
        logging.error(f"Invalid number of samples: {str(e)}")
        raise ValueError("Please enter a valid positive integer for the number of samples")
    logging.info(f"User requested {num_samples} samples")
    print(f"\nPlease upload exactly {num_samples} sample images at once.")
    uploaded = files.upload()
    if len(uploaded) != num_samples:
        logging.error(f"Upload mismatch: expected {num_samples}, got {len(uploaded)}")
        raise ValueError(f"Expected {num_samples} images, but got {len(uploaded)}.")
    image_paths = [(i+1, filename) for i, filename in enumerate(uploaded.keys())]
    mm_per_pixel, calib_data = get_calibration_factor()

    method_display = {
        'reference': 'Reference Object',
        'geometric': 'Geometric (Camera Specs)',
        'manual': 'Manual Entry'
    }.get(calib_data.get('method', ''), 'N/A')

    now = get_ist_now()
    report_metadata = {
        "Software Version": SOFTWARE_VERSION,
        "Operator": operator_name,
        "Date": now.strftime("%Y-%m-%d"),
        "Time": now.strftime("%H:%M:%S"),
        "Camera": calib_data.get('camera_model', 'N/A'),
        "Image Resolution": calib_data.get('image_resolution', 'N/A'),
        "Calibration Method": method_display,
        "Calibration Constant": f"{mm_per_pixel:.4f} mm/pixel"
    }
    logging.info(f"Calibration set to {mm_per_pixel:.4f} mm/pixel")
    print(f"Calibration factor (mm_per_pixel): {mm_per_pixel:.6f}")
    # Unique cache directory for this run
    run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    CACHE_DIR = f"cache_{run_timestamp}"
    os.makedirs(CACHE_DIR, exist_ok=True)
    logging.info(f"Using unique cache directory: {CACHE_DIR}")
    # Initialize result storage
    sample_images = [None] * num_samples
    segmentation_images = [None] * num_samples
    individual_masks_per_sample = [[] for _ in range(num_samples)]
    batch_metrics = []


    def process_image(sample_num, image_path):
        """
        Processes a single image with caching support.


        Args:
            sample_num (int): Sample number.
            image_path (str): Path to the image.


        Returns:
            tuple: (sample_num, metrics, contours, seg_vis, masks).
        """
        cached_metrics = load_metrics(CACHE_DIR, sample_num)
        if cached_metrics:
            logging.info(f"Using cached metrics for sample {sample_num} within this run")
            return sample_num, cached_metrics, None, None, []
        try:
            image = cv2.imread(image_path)
            if image is None:
                raise FileNotFoundError(f"Could not load image {image_path}")
            metrics, contours, seg_vis, masks = process_single_image(image, mm_per_pixel, THRESHOLDS)
            for m in metrics:
                m['Sample'] = sample_num
            save_metrics(CACHE_DIR, sample_num, metrics)
            return sample_num, metrics, contours, seg_vis, masks
        except Exception as e:
            logging.error(f"Error processing sample {sample_num}: {str(e)}")
            return sample_num, [], None, None, []




    # Process images in parallel
    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = {executor.submit(process_image, sn, path): sn for sn, path in image_paths}
        for future in concurrent.futures.as_completed(futures):
            sample_num = futures[future]
            try:
                sn, metrics, contours, seg_vis, masks = future.result()
                batch_metrics.extend(metrics)
                if contours is not None:
                    sample_images[sn-1] = contours
                    segmentation_images[sn-1] = seg_vis
                    individual_masks_per_sample[sn-1] = masks
                logging.info(f"Sample {sn}: {len(metrics)} makhanas analyzed")
            except Exception as e:
                logging.error(f"Error in processing sample {sample_num}: {str(e)}")


    generate_batch_report(batch_metrics, sample_images, segmentation_images, individual_masks_per_sample, mm_per_pixel, THRESHOLDS, num_samples, calib_data, report_metadata)
    generate_batch_excel(batch_metrics, num_samples, mm_per_pixel, report_metadata)
    logging.info("Batch analysis completed successfully!")


In [180]:
# ===========================================================================
# Main Execution
# ===========================================================================


if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
    analyze_batch()

Enter operator name: end
Enter the number of samples to analyze: 1

Please upload exactly 1 sample images at once.


Saving makhana_cam.tif to makhana_cam (18).tif
1. Reference Object (Recommended)
2. Geometric (Camera Specs)
3. Manual Entry
Select method (1-3): 3
Enter camera model (e.g., JAI AD-130GE): jai
Enter known mm/pixel value: 0.1580
Calibration factor (mm_per_pixel): 0.158000
Enter number of rows: 4
Rows found: 4
[137, 131, 134, 145, 127]
[390, 381, 370, 380, 368]
[607, 612, 602, 626, 600]
[831, 833, 819, 827, 794]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>